# Lekcja 6. Wiele kubitów i superpozycja

**Cel:** zrozumieć, w jakiej przestrzeni żyje stan wielu kubitów, skąd bierze się wykładniczy wymiar i czym precyzyjnie jest superpozycja.

## 1. Przestrzeń: iloczyn tensorowy

Stan $n$ kubitów to wektor jednostkowy w

$$\underbrace{\mathbb{C}^2 \otimes \cdots \otimes \mathbb{C}^2}_{n} \;\cong\; \mathbb{C}^{2^n}.$$

> **To było trudne:** wymiar to $2^n$ (dwa do potęgi $n$), a nie $2n$. Dla 3 kubitów to $\mathbb{C}^8$, a nie $\mathbb{R}^8$ (amplitudy są zespolone) ani $\mathbb{C}^4$ (baza ma 8 wektorów, nie 4).

Przy iloczynie tensorowym wymiary się **mnożą** ($2 \cdot 2 \cdot 2 = 8$), przy sumie prostej **dodają** ($2 + 2 + 2 = 6$). Fizyka wybiera iloczyn tensorowy, bo pomiar 3 kubitów ma 8 możliwych wyników, a reguła Borna potrzebuje osobnej amplitudy dla każdego. Nawet klasyczny łączny rozkład trzech bitów potrzebuje 8 liczb, bo bity mogą być skorelowane.

| Kubity | $2n$ | $2^n$ |
|---|---|---|
| 1 | 2 | 2 |
| 2 | 4 | 4 |
| 3 | 6 | **8** |
| 10 | 20 | **1024** |
| 30 | 60 | **1 073 741 824** |

**Intuicja „$n$ niezależnych wektorów w $\mathbb{C}^2$” nie jest błędna, tylko niepełna.** Opisuje jeden szczególny przypadek: stany produktowe $a \otimes b \otimes c$, w których każdy kubit ma własny stan. Większość wektorów w $\mathbb{C}^{2^n}$ tak nie wygląda (lekcja 7).

## 2. Baza i iloczyn Kroneckera

Baza $\mathbb{C}^{2^n}$ to iloczyny wektorów bazowych: $e_a \otimes e_b \otimes e_c$. Indeks to liczba zapisana bitami:

$$e_a \otimes e_b = e_{2a+b}, \qquad e_a \otimes e_b \otimes e_c = e_{4a+2b+c}.$$

Etykieta $|101\rangle$ to więc $e_5 \in \mathbb{C}^8$: jedynka na pozycji $101_2 = 5$.

Bramka działająca na jeden kubit, z „nic nie rób” na pozostałych, to iloczyn Kroneckera z macierzami jednostkowymi, np. X na pierwszym z trzech kubitów:

$$X \otimes I \otimes I = \begin{pmatrix}0 & I_4\\ I_4 & 0\end{pmatrix}.$$

Zamienia górną połowę wektora (stany z pierwszym bitem 0) z dolną (pierwszy bit 1).

In [1]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
GREEN, DARK, LIGHT, GREY, ORANGE = "#44B749", "#1A8A1F", "#96E199", "#8A8A8A", "#E08A00"

e0 = np.array([1, 0], dtype=complex)   # etykieta |0⟩
e1 = np.array([0, 1], dtype=complex)   # etykieta |1⟩

def norm(v):
    return np.sqrt(np.sum(np.abs(v) ** 2))

def show(v, n=None):
    """Wypisz wektor stanu z etykietami bitów (etykieta = pozycja w zapisie dwójkowym)."""
    v = np.asarray(v)
    n = n or int(np.log2(len(v)))
    for k, a in enumerate(v):
        if abs(a) > 1e-12:
            print(f"  pozycja {k} ({format(k, f'0{n}b')}): amplituda {np.round(a, 3)}, P = {abs(a)**2:.3f}")

from functools import reduce
kron = lambda *ms: reduce(np.kron, ms)
X = np.array([[0, 1], [1, 0]])
I2 = np.eye(2, dtype=int)

print("e0 ⊗ e0 =", kron(e0, e0).real.astype(int), "  (|00⟩)")
print("e1 ⊗ e0 ⊗ e1 =", kron(e1, e0, e1).real.astype(int), " (|101⟩ = e5)")

XII = kron(X, I2, I2)
XXX = kron(X, X, X)
etykiety = [format(k, "03b") for k in range(8)]
print("\nX ⊗ I ⊗ I:")
print(XII)
print("działanie:", {etykiety[k]: etykiety[int(np.argmax(XII[:, k]))] for k in range(8)})
print("\nX ⊗ X ⊗ X:", {etykiety[k]: etykiety[int(np.argmax(XXX[:, k]))] for k in range(8)})

e0 ⊗ e0 = [1 0 0 0]   (|00⟩)
e1 ⊗ e0 ⊗ e1 = [0 0 0 0 0 1 0 0]  (|101⟩ = e5)

X ⊗ I ⊗ I:
[[0 0 0 0 1 0 0 0]
 [0 0 0 0 0 1 0 0]
 [0 0 0 0 0 0 1 0]
 [0 0 0 0 0 0 0 1]
 [1 0 0 0 0 0 0 0]
 [0 1 0 0 0 0 0 0]
 [0 0 1 0 0 0 0 0]
 [0 0 0 1 0 0 0 0]]
działanie: {'000': '100', '001': '101', '010': '110', '011': '111', '100': '000', '101': '001', '110': '010', '111': '011'}

X ⊗ X ⊗ X: {'000': '111', '001': '110', '010': '101', '011': '100', '100': '011', '101': '010', '110': '001', '111': '000'}

**Liniowość na superpozycji:** $(X \otimes I \otimes I)\,\frac{1}{\sqrt2}(e_0 + e_3) = \frac{1}{\sqrt2}(e_4 + e_7)$, czyli etykiety $|000\rangle + |011\rangle \mapsto |100\rangle + |111\rangle$.

**Uwaga na kolejność bitów:** PennyLane traktuje kubit 0 jako **pierwszy** bit (tak jak tutaj). Qiskit robi odwrotnie: kubit 0 to **ostatni** bit. Ten sam obwód w Qiskit pokaże „001” zamiast „100”.

**Pamięć symulatora:** stan to $2^n$ liczb zespolonych po 16 bajtów, stąd wykres z prezentacji.

In [2]:
for n in (20, 25, 30, 35, 40, 50):
    bajty = 16 * 2 ** n
    jednostki = ["B", "KiB", "MiB", "GiB", "TiB", "PiB"]
    k = int(np.log2(bajty) // 10)
    print(f"{n} kubitów: 2^{n} amplitud, {bajty / 2 ** (10 * k):.0f} {jednostki[k]}")

20 kubitów: 2^20 amplitud, 16 MiB
25 kubitów: 2^25 amplitud, 512 MiB
30 kubitów: 2^30 amplitud, 16 GiB
35 kubitów: 2^35 amplitud, 512 GiB
40 kubitów: 2^40 amplitud, 16 TiB
50 kubitów: 2^50 amplitud, 16 PiB

## 3. Superpozycja: precyzyjna definicja

> **Superpozycja** to nietrywialna kombinacja liniowa wektorów **danej bazy**: stan ma w tej bazie co najmniej dwie niezerowe współrzędne.

Kilka konsekwencji, które warto mieć ułożone:

- **„Co najmniej dwie” nie zależy od liczby kubitów.** Jedna niezerowa współrzędna oznacza, że stan jest (z dokładnością do fazy) jednym wektorem bazy. Dla 3 kubitów $(e_0 + e_7)/\sqrt2$ ma tylko 2 niezerowe współrzędne z 8 i jest superpozycją.
- **Nie wszystkie współrzędne muszą być niezerowe.** Stan $(1, 1, 1, 1)/2$ to szczególny przypadek: równa superpozycja wszystkich stanów bazowych (punkt startowy Grovera).
- **Stanów bez superpozycji jest fizycznie $2^n$:** wektory bazy, z dokładnością do fazy globalnej. Dla jednego kubitu to bieguny sfery Blocha.
- **To pojęcie zależy od bazy.** Każdy wektor jest kombinacją liniową czegoś (np. $e_0 = (u_+ + u_-)/\sqrt2$). $u_+$ jest superpozycją w bazie $z$, a wektorem bazowym w bazie $x$. Gdy nikt nie mówi „w jakiej bazie”, chodzi o bazę obliczeniową.
- **Uwaga na normalizację:** $(1/2, 1/2)$ nie jest stanem (norma $1/\sqrt2$); poprawny odpowiednik to $(1/\sqrt2, 1/\sqrt2)$.

„Kubit w superpozycji jest jednocześnie 0 i 1” to mylące zdanie. Lepiej: kubit jest w innym stanie niż $e_0$ i niż $e_1$, a pomiar w bazie $z$ daje losowy wynik.

**Zasada superpozycji** (drugie znaczenie słowa) to postulat: każda znormalizowana kombinacja liniowa stanów jest stanem, czyli stany tworzą przestrzeń wektorową.

In [3]:
def czy_superpozycja(v):
    return np.count_nonzero(np.abs(v) > 1e-12) >= 2

przyklady = {
    "(0,1,0,0)  |01⟩":          np.array([0, 1, 0, 0]),
    "(1,0,0,1)/√2  Bell":       np.array([1, 0, 0, 1]) / np.sqrt(2),
    "(1,0,1,0)/√2  |+⟩⊗|0⟩":    np.array([1, 0, 1, 0]) / np.sqrt(2),
    "(1,1,1,1)/2  |+⟩⊗|+⟩":     np.array([1, 1, 1, 1]) / 2,
    "(e0+e7)/√2, 3 kubity":     (np.eye(8)[0] + np.eye(8)[7]) / np.sqrt(2),
}
for nazwa, v in przyklady.items():
    print(f"{nazwa:>26}: niezerowe = {np.count_nonzero(v)}, superpozycja? {czy_superpozycja(v)}")

           (0,1,0,0)  |01⟩: niezerowe = 1, superpozycja? False
        (1,0,0,1)/√2  Bell: niezerowe = 2, superpozycja? True
     (1,0,1,0)/√2  |+⟩⊗|0⟩: niezerowe = 2, superpozycja? True
      (1,1,1,1)/2  |+⟩⊗|+⟩: niezerowe = 4, superpozycja? True
      (e0+e7)/√2, 3 kubity: niezerowe = 2, superpozycja? True

## Podsumowanie

> Stan $n$ kubitów to wektor jednostkowy w $\mathbb{C}^{2^n} = (\mathbb{C}^2)^{\otimes n}$. Baza to wektory $e_k$, gdzie $k$ zapisane dwójkowo to ciąg bitów (kodowanie one-hot). Bramki na pojedynczych kubitach to iloczyny Kroneckera z macierzami jednostkowymi. Superpozycja to nietrywialna kombinacja liniowa wektorów danej bazy: co najmniej dwie niezerowe współrzędne, niezależnie od liczby kubitów.

## Zadania

1. Zapisz $e_1 \otimes e_1 \otimes e_0$ jako wektor w $\mathbb{C}^8$. Jaka to etykieta?
2. Jak wygląda macierz $I \otimes X$ (X na drugim z dwóch kubitów)? Na co przechodzi $e_0 = |00\rangle$?
3. Ile niezerowych współrzędnych ma stan $H\!\otimes\!H\!\otimes\!H\, e_0$? Czy jest superpozycją?